# Library Installation and Import

In [ ]:
if (!requireNamespace("Seurat", quietly = TRUE)) install.packages("Seurat")
if (!requireNamespace("BPCells", quietly = FALSE)) remotes::install_github("bnprks/BPCells")
if (!requireNamespace("fs", quietly = TRUE)) install.packages("fs")
if (!requireNamespace("harmony", quietly = TRUE)) install.packages("harmony")

library(future)

# plan("multicore", workers = 24) # Mac Pro 6.1
# plan(workers = 36) # iMac Pro 1.1
# plan("multicore", workers = 8) # 4 core Intel CPU and M1 have 8 threads
# plan("multicore", workers = 12) # 6 core Intel CPU has 12 threads
plan("multicore", workers = 10) # 10 core M1 Pro CPU has 10 threads

# Set RAM Size to 3/4 of total RAM


# options(future.globals.maxSize = 48000 * 1024^2) # Mac Pros have 64 Gb
options(future.globals.maxSize = 16000 * 1024^2) # iMacs and MacBook Pros have 16 Gb

future.seed=NULL # Removes future-generated statistical errors

library(Seurat)
library(BPCells)
library(dplyr)
library(fs)
library(Matrix)
library(ggplot2)
library(viridis)
library(harmony)
library(fs)
library(stringr)
library(SoupX)
library(DropletUtils)

options(Seurat.object.assay.version = "v5")
base_data_dir <- "./Data"
output_bp_dir <- "./Processed_BPCells" 

dir_create(output_bp_dir)

# Preprocessing 

In [ ]:
# Step 1: Scan Data
find_batches <- function(root_dir) {
  message(paste("Scanning in:", path_abs(root_dir)))
  
  all_dirs <- dir_ls(root_dir, recurse = TRUE, type = "directory")
  iso_anchors <- grep("filtered_isoform_matrix$", all_dirs, value = TRUE)
  
  if (length(iso_anchors) == 0) {
    warning("No folders ending in 'filtered_isoform_matrix' found.")
    return(list())
  }
  
  message(paste("Found", length(iso_anchors), "potential isoform folders."))
  batch_list <- list()
  
  for (iso_path in iso_anchors) {
    batch_dir <- path_dir(iso_path)
    parts <- path_split(batch_dir)[[1]]
    n <- length(parts)
    
    batch_name <- parts[n]
    experiment_name <- parts[n-1]
    batch_id <- paste(experiment_name, batch_name, sep = "_")
    
    rna_filt_path <- path(batch_dir, "filtered_feature_bc_matrix")
    rna_raw_path <- path(batch_dir, "raw_feature_bc_matrix")
    
    if (!dir_exists(rna_filt_path) || !dir_exists(rna_raw_path)) {
      message(paste("Skipping", batch_id, "- Isoforms found, but filtered or raw RNA matrix missing."))
      next
    }
    
    batch_list[[batch_id]] <- list(
      id = batch_id,
      bio_group = experiment_name,
      batch_name = batch_name,
      iso_filt = iso_path,
      rna_filt = rna_filt_path,
      rna_raw = rna_raw_path 
    )
  }
  return(batch_list)
}

batches <- find_batches("./Data")
print(paste("SUCCESS: Found", length(batches), "batches ready for processing."))

In [ ]:
# Step 2: Processing Loop with SoupX
seurat_list <- list()

for (batch in batches) {
  message(paste0("\n>>> Processing Batch: ", batch$id))
  
  message("  [RNA] Loading Raw & Filtered Matrices...")
  mat_rna_raw <- Read10X(batch$rna_raw)
  mat_rna <- Read10X(batch$rna_filt)
  
  message("  [Isoform] Loading Filtered Matrix...")
  mat_iso <- Read10X(batch$iso_filt)

  common_cells <- intersect(colnames(mat_rna), colnames(mat_iso))
  
  if (length(common_cells) < 100) {
    warning("Too few matching cells! Skipping batch.")
    next
  }
  
  mat_rna <- mat_rna[, common_cells]
  mat_iso <- mat_iso[, common_cells]

  message("  [SoupX] Estimating ambient RNA...")
  
  # Initialize SoupChannel
  sc <- SoupChannel(tod = mat_rna_raw, toc = mat_rna)
  
  # Quick standard clustering on the filtered data
  tmp_sobj <- CreateSeuratObject(counts = mat_rna)
  tmp_sobj <- NormalizeData(tmp_sobj, verbose = FALSE)
  tmp_sobj <- FindVariableFeatures(tmp_sobj, verbose = FALSE)
  tmp_sobj <- ScaleData(tmp_sobj, verbose = FALSE)
  tmp_sobj <- RunPCA(tmp_sobj, verbose = FALSE)
  tmp_sobj <- FindNeighbors(tmp_sobj, dims = 1:15, verbose = FALSE)
  tmp_sobj <- FindClusters(tmp_sobj, resolution = 0.5, verbose = FALSE)
  
  # Feed the clusters into SoupChannel
  sc <- setClusters(sc, setNames(tmp_sobj$seurat_clusters, colnames(tmp_sobj)))
  
  # TF-IDF contamination estimation
  sc <- tryCatch({
    message("  [SoupX] Running autoEstCont...")
    autoEstCont(sc, doPlot = FALSE)
  }, error = function(e) {
    message("  [SoupX] WARNING: Auto-estimation failed (sample likely too homogenous). Applying safe 5% background.")
    setContaminationFraction(sc, 0.05)
  })
  
  message(paste("  [SoupX] Estimated Contamination Fraction:", round(sc$fit$rhoEst * 100, 2), "%"))
  
  # Scrub the matrix
  message("  [SoupX] Scrubbing matrix...")
  mat_rna_clean <- adjustCounts(sc, roundToInt = TRUE)
  
  # Free up RAM
  rm(mat_rna_raw, mat_rna, tmp_sobj, sc)
  gc()
  
  bp_rna_path <- path(output_bp_dir, batch$id, "RNA")
  bp_iso_path <- path(output_bp_dir, batch$id, "Isoform")
  
  message("  [BPCells] Writing CLEANED matrices to disk...")
  write_matrix_dir(mat = mat_rna_clean, dir = bp_rna_path, overwrite = TRUE)
  write_matrix_dir(mat = mat_iso, dir = bp_iso_path, overwrite = TRUE)
  
  bp_rna_ptr <- open_matrix_dir(dir = bp_rna_path)
  bp_iso_ptr <- open_matrix_dir(dir = bp_iso_path)
  
  sobj <- CreateSeuratObject(counts = bp_rna_ptr, assay = "RNA", project = batch$batch_name)
  sobj[["Isoform"]] <- CreateAssay5Object(counts = bp_iso_ptr)
  
  sobj$BioGroup <- batch$bio_group
  sobj$BatchID <- batch$batch_name
  sobj$Batch <- paste(batch$bio_group, batch$batch_name, sep = "_")
  
  seurat_list[[batch$id]] <- sobj
  
  rm(mat_rna_clean, mat_iso, sobj, bp_rna_ptr, bp_iso_ptr)
  gc()
}

In [ ]:
# Step 3: Merging
message("\nDone processing all batches.")

if (length(seurat_list) > 0) {
  message("Merging into final object...")
  
  merged_obj <- merge(
    x = seurat_list[[1]],
    y = seurat_list[-1],
    add.cell.ids = names(seurat_list)
  )
  
  merged_obj <- JoinLayers(merged_obj)
  print(merged_obj)
  
} else {
  warning("No valid batches were processed.")
}

In [ ]:
DefaultAssay(merged_obj) <- "Isoform"

In [ ]:
merged_obj[["percent.mt"]] <- PercentageFeatureSet(merged_obj,
                                                   pattern = "^mt-",
                                                   assay = "Isoform")

merged_obj[["percent.hb"]] <- PercentageFeatureSet(merged_obj, 
                                                   pattern = "^Hb[a-z]+",
                                                   assay = "Isoform")

In [ ]:
seurat_object <- merged_obj

In [ ]:
Idents(seurat_object) <- "Batch"

In [ ]:
# Visualize QC metrics as a violin plot
VlnPlot(seurat_object, features = c("nFeature_RNA", "nCount_RNA"), ncol = 2)
VlnPlot(seurat_object, features = c("nFeature_Isoform", "nCount_Isoform"), ncol = 2)
VlnPlot(seurat_object, features = c("percent.mt", "percent.hb"), ncol = 2)

In [ ]:
seurat_object
head(seurat_object@meta.data)

In [ ]:
FeatureScatter(seurat_object, feature1 = "nCount_RNA", feature2 = "percent.mt")
FeatureScatter(seurat_object, feature1 = "nCount_Isoform", feature2 = "percent.mt")
FeatureScatter(seurat_object, feature1 = "nCount_RNA", feature2 = "nFeature_RNA")
FeatureScatter(seurat_object, feature1 = "nCount_Isoform", feature2 = "nFeature_Isoform")

In [ ]:
saveRDS(seurat_object, "BPcells Counts.rds")

In [ ]:
subset_seurat_obj <- subset(seurat_object, subset = nFeature_Isoform > 0 & nCount_Isoform < 60000
                            & percent.mt < 7.5 
                           )
subset_seurat_obj

In [ ]:
FeatureScatter(subset_seurat_obj, feature1 = "nCount_Isoform", feature2 = "percent.mt")
FeatureScatter(subset_seurat_obj, feature1 = "nCount_Isoform", feature2 = "nFeature_Isoform")

In [ ]:
output_clean_mtx_dir <- "./Cleaned_MTX_For_Python"
dir_create(output_clean_mtx_dir)

batch_ids <- unique(subset_seurat_obj$Batch)

message(paste("Exporting", length(batch_ids), "batches for Scrublet..."))

for (b in batch_ids) {
  message(paste("  Exporting Batch:", b))
  
  # Get the barcodes
  cells_in_batch <- colnames(subset_seurat_obj)[subset_seurat_obj$Batch == b]
  
  # Extract counts specifically from the RNA assay
  bp_mat <- subset_seurat_obj[["RNA"]]$counts[, cells_in_batch]
  rna_genes <- rownames(subset_seurat_obj[["RNA"]])
  
  # Convert to standard sparse matrix
  mat_batch <- as(bp_mat, "dgCMatrix")

  rownames(mat_batch) <- rna_genes
  colnames(mat_batch) <- cells_in_batch
  
  batch_out_path <- fs::path(output_clean_mtx_dir, b)

  write10xCounts(
    x = mat_batch, 
    path = batch_out_path, 
    overwrite = TRUE,
    version = "3" 
  )
  
  rm(mat_batch, bp_mat); gc()
}

message("Done! Ready for Python.")

In [ ]:
# Remove Doublets

scrublet_res <- read.csv("Python_Scrublet_Results.csv", row.names = 1)
subset_seurat_obj <- AddMetaData(subset_seurat_obj, metadata = scrublet_res)
print(table(subset_seurat_obj$Doublet_Status))
subset_seurat_obj <- subset(subset_seurat_obj, subset = Doublet_Status == "Singlet")

In [ ]:
# 40k - in the grim darkness of lipid droplets there are only doublets

# fellas, please, use BD Rhapsody, I beg ye, Sample Tags are awesome btw

In [ ]:
message("Pre-join class: ", class(subset_seurat_obj[["RNA"]]$counts))

subset_seurat_obj <- JoinLayers(subset_seurat_obj)

message("Post-join class: ", class(subset_seurat_obj[["RNA"]]$counts))

In [ ]:
message("Post-join class: ", class(subset_seurat_obj[["Isoform"]]$counts))

In [ ]:
Layers(subset_seurat_obj[["RNA"]])

In [ ]:
# GEX noise removal & GEX norm (logNorm for Isoform, RAM-optimized SCT for RNA)

plan("sequential")

rna_counts <- subset_seurat_obj[["RNA"]]$counts
keep_rna <- rownames(rna_counts)[Matrix::rowSums(rna_counts > 0) >= 30]

iso_counts <- subset_seurat_obj[["Isoform"]]$counts
keep_iso <- rownames(iso_counts)[Matrix::rowSums(iso_counts > 0) >= 30]

rna_assay_filtered <- subset(subset_seurat_obj[["RNA"]], features = keep_rna)
iso_assay_filtered <- subset(subset_seurat_obj[["Isoform"]], features = keep_iso)

meta_data_saved <- subset_seurat_obj@meta.data

subset_seurat_obj <- CreateSeuratObject(
  counts = rna_assay_filtered$counts,
  assay = "RNA"
)

subset_seurat_obj[["Isoform"]] <- CreateAssay5Object(
  counts = iso_assay_filtered$counts
)

subset_seurat_obj@meta.data <- meta_data_saved

rm(rna_assay_filtered, iso_assay_filtered, meta_data_saved, rna_counts, iso_counts)
gc()

DefaultAssay(subset_seurat_obj) <- "Isoform"

subset_seurat_obj <- NormalizeData(
  subset_seurat_obj, 
  normalization.method = "LogNormalize", 
  scale.factor = 10000
)

subset_seurat_obj <- FindVariableFeatures(
  subset_seurat_obj, 
  selection.method = "vst", 
  nfeatures = 2000
)

subset_seurat_obj <- ScaleData(subset_seurat_obj)

DefaultAssay(subset_seurat_obj) <- "RNA"

subset_seurat_obj[["RNA"]] <- split(subset_seurat_obj[["RNA"]], f = subset_seurat_obj$Batch)

subset_seurat_obj <- SCTransform(
  subset_seurat_obj,
  assay = "RNA",
  new.assay.name = "SCT",
  vars.to.regress = "percent.mt",
  vst.flavor = "v2",
  variable.features.n = 3000,
  ncells = 3000,
  verbose = TRUE
)

In [ ]:
Seurat <- subset_seurat_obj

In [ ]:
DefaultAssay(Seurat) <- "SCT"
Seurat <- JoinLayers(Seurat)
DefaultAssay(Seurat) <- "Isoform"
Seurat <- JoinLayers(Seurat)

In [ ]:
DefaultAssay(Seurat) <- "SCT"
junk_pattern <- "^mt-|^Rp[sl]|^Hsp|^Sno|^Snr|^Rn7s|^Gm[0-9]"
current_hvgs <- VariableFeatures(Seurat)
junk_hvgs <- grep(pattern = junk_pattern, x = current_hvgs, value = TRUE)
clean_hvgs <- setdiff(current_hvgs, junk_hvgs)
VariableFeatures(Seurat) <- clean_hvgs

message(paste("Removed", length(junk_hvgs), "junk genes from Variable Features."))

Seurat <- RunPCA(Seurat, features = clean_hvgs, verbose = FALSE)

ElbowPlot(Seurat, ndims = 50, reduction = "pca")

In [ ]:
# Correct Batch Effects via Harmony
Seurat <- RunHarmony(Seurat, 
                                group.by.vars = c("Batch"),
                                reduction = "pca", assay.use = "SCT", plot_convergence = FALSE, 
                                project.dim = F, reduction.save = "mRNA_harmony")

In [ ]:
# Harmony Elbow Plot
ElbowPlot(Seurat, ndims = 50, reduction = "mRNA_harmony")

In [ ]:
# UMAP
dims = 27

Seurat <- FindNeighbors(Seurat, dims = 1:dims
                        , reduction = "mRNA_harmony"
                       )
Seurat <- RunUMAP(Seurat, dims = 1:dims 
                 , reduction = "mRNA_harmony"
                 )

In [ ]:
Seurat <- FindClusters(Seurat, resolution = 0.4)
DimPlot(Seurat, label=TRUE, reduction = "umap")

ggsave( 
  "Clusters Raw.png",
  plot   = last_plot(),
  width  = 22,
  height = 18,
  units  = "cm",
  dpi    = 600,
  limitsize = TRUE,
  bg     = "white"
)

In [ ]:
saveRDS(Seurat, file = "Seurat_Clusters.rds")

# Cluster Identification

In [ ]:
Seurat <- readRDS('Seurat_Clusters.rds')

In [ ]:
Seurat <- PrepSCTFindMarkers(Seurat)

In [ ]:
library(dplyr)

DefaultAssay(Seurat) <- "SCT"

markers_rna <- FindAllMarkers(
  Seurat,
  only.pos        = TRUE,
  min.pct         = 0.33,
  logfc.threshold = 0.5,
  test.use        = "wilcox"
)

if (!"avg_log2FC" %in% colnames(markers_rna) && "avg_logFC" %in% colnames(markers_rna)) {
  markers_rna$avg_log2FC <- markers_rna$avg_logFC
}

top_rna <- markers_rna %>%
  filter(p_val_adj < 0.05) %>%
  group_by(cluster) %>%
  arrange(desc(avg_log2FC), .by_group = TRUE) %>%
  slice_head(n = 200) %>%
  ungroup()

write.csv(top_rna, file = "Significant_RNA_Markers_Top200.csv", row.names = FALSE)

In [ ]:
cluster_names <- c(
  "0"  = "Erythroid Cells",
  "1"  = "Erythroid Cells",
  "8"  = "Erythroid Cells",
  "10" = "Erythroid Cells",
  "4"  = "Erythroid Cells",
  "14" = "Erythroid Cells",

  "12" = "Hofbauer Cells",
  "9"  = "Macrophages (APCs)",
  "15" = "Macrophages (Inflammatory)",
  "17" = "Macrophages (Remodeling)",
  "25" = "Monocytes",
  
  "5"  = "Myofibroblasts",
  "13" = "Fibroblasts",
  
  "20" = "Endothelial Cells",
  "26" = "Pericytes",
  
  "2"  = "Labyrinth Trophoblast",
  "11" = "Spongiotrophoblasts",
  "16" = "Giant Cells",
  "22" = "Syncytiotrophoblasts",
  "29" = "Glycogen Trophoblasts",
  
  "6"  = "Decidua Early",
  "28" = "Decidua Late",
  "30" = "Glandular Epithelium",
  
  "7"  = "Neutrophils",
  "18" = "T-cells",
  "24" = "uNK-cells",
  "27" = "Megakaryocytes",
  "21" = "Labyrinthine Mesenchyme",
  
  "3"  = "Yolk Sac Endoderm",
  "19" = "Fetal Neural Tissue",
  "23" = "Fetal Hepatocytes"
)

Seurat <- RenameIdents(Seurat, cluster_names)
Seurat$CellType <- Idents(Seurat)

target_order <- c(
  "Endothelial Cells", "Pericytes", "Labyrinthine Mesenchyme", "Syncytiotrophoblasts", 
  "Giant Cells", "Glycogen Trophoblasts", "Spongiotrophoblasts", "Labyrinth Trophoblast", 
  "Fibroblasts", "Myofibroblasts", "Decidua Early", "Decidua Late", "Glandular Epithelium", 
  "Macrophages (Remodeling)", "Macrophages (Inflammatory)", "Macrophages (APCs)", 
  "Hofbauer Cells", "Monocytes", "T-cells", "uNK-cells", "Neutrophils", "Megakaryocytes", 
  "Erythroid Cells"
)

Seurat$CellType <- factor(Seurat$CellType, levels = target_order)

In [ ]:
fetal_contamination <- c("Yolk Sac Endoderm", "Fetal Neural Tissue", "Fetal Hepatocytes")
Seurat <- subset(Seurat, idents = fetal_contamination, invert = TRUE)

In [ ]:
DimPlot(
  Seurat, 
  reduction = "umap", 
  # group.by = "CellType", 
  label = TRUE, 
  repel = TRUE,
  label.size = 4
)

ggsave( 
  "Clusters.png",
  plot   = last_plot(),
  width  = 32.7,
  height = 20,
  units  = "cm",
  dpi    = 600,
  limitsize = TRUE,
  bg     = "white"
)

In [ ]:
table(Seurat$CellType)

In [ ]:
Seurat$UMAP_1 <- Embeddings(Seurat, "umap")[, 1]
Seurat$UMAP_2 <- Embeddings(Seurat, "umap")[, 2]

cells_to_keep <- WhichCells(Seurat, expression = !((UMAP_1 > -10 & UMAP_1 < -2.5) & (UMAP_2 > -15 & UMAP_2 < -6)))

Seurat <- Seurat[, cells_to_keep]

In [ ]:
Seurat$CellType <- factor(Seurat$CellType, levels = rev(levels(Seurat$CellType)))
Idents(Seurat) <- "CellType"

In [ ]:
DimPlot(
  Seurat, 
  reduction = "umap", 
  # group.by = "CellType", 
  label = TRUE, 
  repel = TRUE,
  # label.box = TRUE,
  label.size = 4
)

ggsave( 
  "Clusters.png",
  plot   = last_plot(),
  width  = 40,
  height = 26.5,
  units  = "cm",
  dpi    = 600,
  limitsize = TRUE,
  bg     = "white"
)

In [ ]:
# Clean the residual Globin Soup 

seurat_obj <- Seurat 

all_genes <- c()
for (assay_name in names(seurat_obj@assays)) {
  genes <- as.character(rownames(seurat_obj[[assay_name]]))
  all_genes <- c(all_genes, genes)
}
all_genes <- unique(all_genes)

# Isolate Erythroid genes
ery_genes <- all_genes[grep("^Hb[ab]-|^Alas2$|^Slc25a37$|^Gypa$|^Trim10$|^Hemgn$", all_genes, ignore.case = TRUE)]

clean_layer <- function(mat, genes, cells) {
  if (is.null(mat)) return(mat)
  
  if (!inherits(mat, "dgCMatrix")) {
    mat <- as(mat, "dgCMatrix")
  }
  
  rn <- rownames(mat)
  cn <- colnames(mat)
  
  g <- intersect(genes, rn)
  c <- intersect(cells, cn)
  
  if (length(g) == 0 || length(c) == 0) return(mat)
  
  idx <- which(rn %in% g)
  jdx <- which(cn %in% c)
  
  mat[idx, jdx] <- 0
  
  return(mat)
}

clean_soup_v5 <- function(obj, assay_name, genes, protect_clusters) {
  if (!(assay_name %in% names(obj@assays))) return(obj)
  
  cells_to_clean <- colnames(obj)[!(obj$CellType %in% protect_clusters)]
  
  existing_genes <- intersect(genes, rownames(obj[[assay_name]]))
  if (length(existing_genes) == 0) return(obj)
  
  layers <- Layers(obj, assay = assay_name)
  
  if ("data" %in% layers) {
    mat <- tryCatch(LayerData(obj, assay = assay_name, layer = "data"), error=function(e) NULL)
    mat <- clean_layer(mat, existing_genes, cells_to_clean)
    if (!is.null(mat)) LayerData(obj, assay = assay_name, layer = "data") <- mat
  }
  
  if ("counts" %in% layers) {
    mat <- tryCatch(LayerData(obj, assay = assay_name, layer = "counts"), error=function(e) NULL)
    mat <- clean_layer(mat, existing_genes, cells_to_clean)
    if (!is.null(mat)) LayerData(obj, assay = assay_name, layer = "counts") <- mat
  }
  
  return(obj)
}

# Execute 
protected_groups <- c("Erythroid Cells")

seurat_obj <- clean_soup_v5(seurat_obj, "RNA", ery_genes, protected_groups)
seurat_obj <- clean_soup_v5(seurat_obj, "SCT", ery_genes, protected_groups)
seurat_obj <- clean_soup_v5(seurat_obj, "Isoform", ery_genes, protected_groups)

gc()

Seurat <- seurat_obj

In [ ]:
head(Seurat@meta.data)

In [ ]:
junk_cols <- c("seurat_clusters", "SCT_snn_res.0.4", "UMAP_1","UMAP_2")

Seurat@meta.data <- Seurat@meta.data[, !(colnames(Seurat@meta.data) %in% junk_cols)]

In [ ]:
Seurat

head(Seurat@meta.data)

In [ ]:
saveRDS(Seurat, file = "Seurat_Clusters_Annotated.rds")

# Analysis

In [ ]:
Seurat <- readRDS("Seurat_Clusters_Annotated.rds")

In [ ]:
# Erythroid Cell subclustering data

Ery <- readRDS("Erythroid_Cells.rds")

In [ ]:
Seurat$CellTypeDeep <- as.character(Seurat$CellType)

ery_metadata <- Ery@meta.data
Seurat@meta.data[rownames(ery_metadata), "CellTypeDeep"] <- as.character(ery_metadata$Erythroid)

Seurat <- subset(Seurat, subset = CellTypeDeep != "Erythroid Cells")

In [ ]:
target_order <- c(
  "Endothelial Cells", "Pericytes", "Labyrinthine Mesenchyme", "Syncytiotrophoblasts", 
  "Giant Cells", "Glycogen Trophoblasts", "Spongiotrophoblasts", "Labyrinth Trophoblast", 
  "Fibroblasts", "Myofibroblasts", "Decidua Early", "Decidua Late", "Glandular Epithelium", 
  "Macrophages (Remodeling)", "Macrophages (Inflammatory)", "Macrophages (APCs)", 
  "Hofbauer Cells", "Monocytes", "T-cells", "uNK-cells", "Neutrophils", "Megakaryocytes", 
  "Erythroid Cells (Mature)",
  "Erythroid Cells (Fetal)",
  "NK-cell-like Erythroid Cells",
  "APC-like Erythroid Cells",
  "Neutrophil-like Erythroid Cells"
)

Seurat$CellTypeDeep <- factor(Seurat$CellTypeDeep, levels = rev(target_order))

In [ ]:
p <- DimPlot(
  Seurat, 
  reduction = "umap", 
  group.by = "CellTypeDeep", 
  label = TRUE, 
  repel = TRUE,
  label.size = 4
)

ggsave( 
  filename = "Deep Clusters.png",
  plot     = p,
  width    = 40,
  height   = 26.5,
  units    = "cm",
  dpi      = 600,
  limitsize = TRUE,
  bg       = "white"
)

In [ ]:
Idents(Seurat) <- "CellTypeDeep"
saveRDS(Seurat, "Seurat_Deep_Clusters_Annotated.rds")

In [ ]:
Seurat <- readRDS("Seurat_Deep_Clusters_Annotated.rds")

In [ ]:
head(Seurat@meta.data)

In [ ]:
Seurat@reductions

In [ ]:
# Export for CellChat

sct_data_in_memory <- as(GetAssayData(Seurat, assay = "SCT", layer = "data"), "dgCMatrix")

sct_counts_in_memory <- as(GetAssayData(Seurat, assay = "SCT", layer = "counts"), "dgCMatrix")

Seurat_Slim <- CreateSeuratObject(counts = sct_counts_in_memory, assay = "SCT")
Seurat_Slim <- SetAssayData(Seurat_Slim, assay = "SCT", layer = "data", new.data = sct_data_in_memory)

Seurat_Slim@meta.data <- Seurat@meta.data

VariableFeatures(Seurat_Slim, assay = "SCT") <- VariableFeatures(Seurat, assay = "SCT")

saveRDS(Seurat_Slim, "Seurat_For_CellChat.rds")

message("Slim object saved.")

In [ ]:
# Export for pySCENIC

library(magrittr)
library(SingleCellExperiment)
library(SCopeLoomR)

exprMat <- Seurat[["SCT"]]$data
cellInfo <- Seurat@meta.data


loci1 <- which(rowSums(exprMat) > 1*.0005*ncol(exprMat))
exprMat_filter <- exprMat[loci1, ]
exprMat_filter

add_cell_annotation <- function(loom, cellAnnotation)
{
  cellAnnotation <- data.frame(cellAnnotation)
  if(any(c("nGene", "nUMI") %in% colnames(cellAnnotation)))
  {
    warning("Columns 'nGene' and 'nUMI' will not be added as annotations to the loom file.")
    cellAnnotation <- cellAnnotation[,colnames(cellAnnotation) != "nGene", drop=FALSE]
    cellAnnotation <- cellAnnotation[,colnames(cellAnnotation) != "nUMI", drop=FALSE]
  }
  
  if(ncol(cellAnnotation)<=0) stop("The cell annotation contains no columns")
  if(!all(get_cell_ids(loom) %in% rownames(cellAnnotation))) stop("Cell IDs are missing in the annotation")
  
  cellAnnotation <- cellAnnotation[get_cell_ids(loom),,drop=FALSE]
  # Add annotation
  for(cn in colnames(cellAnnotation))
  {
    add_col_attr(loom=loom, key=cn, value=cellAnnotation[,cn])
  }
  
  invisible(loom)
}

loom <- build_loom("Seurat.loom", dgem=exprMat_filter)
loom <- add_cell_annotation(loom, cellInfo)
close_loom(loom)

DefaultAssay(Seurat) <- "SCT"

In [ ]:
auc_raw <- t(read.csv("pySCENIC-AUC-Raw.csv", row.names = 1))
auc_bin <- t(read.csv("pySCENIC-AUC-Binary.csv", row.names = 1))

rownames(auc_raw) <- gsub("\\(\\+\\)", "", rownames(auc_raw))
rownames(auc_bin) <- gsub("\\(\\+\\)", "", rownames(auc_bin))

Seurat[["AUC"]] <- CreateAssayObject(counts = auc_raw)
Seurat[["AUC_Binary"]] <- CreateAssayObject(counts = auc_bin)

In [ ]:
names(Seurat)

In [ ]:
# Idents(Seurat) <- "CellType"

In [ ]:
biogroup_order <- c(
  "E19.5 Semi-Allo", 
  "E19.5 Syn", 
  "E12.5 Syn", 
  "E12.5 Semi-Allo"
)

Seurat$BioGroup <- factor(
  Seurat$BioGroup, 
  levels = biogroup_order
)

levels(Seurat$BioGroup)

In [ ]:
DefaultAssay(Seurat) <- "SCT"
Seurat
DefaultAssay(Seurat) <- "Isoform"
Seurat
DefaultAssay(Seurat) <- "AUC"
Seurat
head(Seurat@meta.data)

In [ ]:
library(Seurat)
library(dplyr)
library(ggplot2)
library(ggrepel)

# Params
logfc_thresh <- 1.0
qval_thresh <- 0.01
min_pct_val <- 0.33

# Assays
assays_to_run <- c("SCT", "Isoform", "AUC")

# List of comparisons
comparisons <- list(
  c("E12.5 Semi-Allo", "E12.5 Syn"),
  c("E19.5 Semi-Allo", "E19.5 Syn"),
  c("E19.5 Semi-Allo", "E12.5 Semi-Allo"),
  c("E19.5 Syn",       "E12.5 Syn")
)

# Metadata preparation
seurat_obj$CellType_BioGroup <- paste0(seurat_obj$CellTypeDeep, "_", seurat_obj$BioGroup)
Idents(seurat_obj) <- "CellType_BioGroup"

# Extract all unique cell types, excluding NAs
cell_types <- unique(seurat_obj$CellTypeDeep)
cell_types <- cell_types[!is.na(cell_types)]

# Main loop
for (assay in assays_to_run) {
  
  if (!(assay %in% names(seurat_obj@assays))) {
    message(sprintf("Assay %s not found in the object. Skipping.", assay))
    next
  }
  
  # Set the active assay
  DefaultAssay(seurat_obj) <- assay
  
  for (ct in cell_types) {
    
    dir_path <- file.path(".", "DGE", ct)
    dir.create(dir_path, recursive = TRUE, showWarnings = FALSE)
    
    for (comp in comparisons) {
      
      group1 <- comp[1]
      group2 <- comp[2]
      
      ident1 <- paste0(ct, "_", group1)
      ident2 <- paste0(ct, "_", group2)

      count1 <- sum(seurat_obj$CellType_BioGroup == ident1, na.rm = TRUE)
      count2 <- sum(seurat_obj$CellType_BioGroup == ident2, na.rm = TRUE)
      
      if (count1 < 3 || count2 < 3) {
        message(sprintf("Skipping: %s (%s vs %s) in %s - insufficient cells (n1=%d, n2=%d).", 
                        ct, group1, group2, assay, count1, count2))
        next
      }
      
      message(sprintf("Calculating: %s | %s vs %s | Assay: %s", ct, group1, group2, assay))
      
      # tryCatch wrapper to prevent the loop from crashing
      dge <- tryCatch({
        # Seurat v5 specific fix for SCT assay
        if (assay == "SCT") {
          FindMarkers(seurat_obj, 
                      ident.1 = ident1, 
                      ident.2 = ident2, 
                      logfc.threshold = 0, 
                      min.pct = min_pct_val,
                      assay = assay,
                      recorrect_umi = FALSE)
        } else {
          FindMarkers(seurat_obj, 
                      ident.1 = ident1, 
                      ident.2 = ident2, 
                      logfc.threshold = 0, 
                      min.pct = min_pct_val,
                      assay = assay)
        }
      }, error = function(e) {
        message("Error during calculation: ", e$message)
        return(NULL)
      })
      
      if (is.null(dge) || nrow(dge) == 0) next
      
      # Process results
      dge$gene <- rownames(dge)
      dge$gene <- gsub("\\.+$", "", dge$gene)
      
      # Protect against infinity
      min_nonzero_q <- min(dge$p_val_adj[dge$p_val_adj > 0], na.rm = TRUE)
      if (is.infinite(min_nonzero_q) || is.na(min_nonzero_q)) min_nonzero_q <- 1e-300
      dge$p_val_adj_plot <- ifelse(dge$p_val_adj == 0, min_nonzero_q, dge$p_val_adj)
      
      # Significance labeling for coloring
      dge$Significance <- "Non-Significant"
      dge$Significance[dge$avg_log2FC > logfc_thresh & dge$p_val_adj < qval_thresh] <- "Up"
      dge$Significance[dge$avg_log2FC < -logfc_thresh & dge$p_val_adj < qval_thresh] <- "Down"

      # Save CSV
      g1_safe <- gsub(" ", "", group1)
      g2_safe <- gsub(" ", "", group2)
      
      base_name <- paste0(assay, "_", g1_safe, "vs", g2_safe)
      csv_file <- file.path(dir_path, paste0(base_name, ".csv"))
      
      dge_filtered <- dge %>% filter(Significance != "Non-Significant")
      write.csv(dge_filtered, csv_file, row.names = FALSE) 
      
      # Generate Volcano Plot
      plot_file <- file.path(dir_path, paste0(base_name, ".jpg"))
      
      plot_data <- dge %>% filter(p_val_adj < 1)
      
      if (nrow(plot_data) == 0) next
      
      # Plot in bwr style
      p <- ggplot(plot_data, aes(x = avg_log2FC, y = -log10(p_val_adj_plot), color = Significance)) +
        geom_point(alpha = 0.6, size = 1.5) +
        scale_color_manual(values = c("Down" = "blue", "Non-Significant" = "grey80", "Up" = "red")) +
        theme_minimal(base_size = 14) +
        labs(
          title = paste(ct, "|", group1, "vs", group2),
          subtitle = paste("Assay:", assay),
          x = "log2(Fold Change)",
          y = "-log10(Adjusted P-value)"
        ) +
        geom_hline(yintercept = -log10(qval_thresh), linetype = "dashed", color = "black") +
        geom_vline(xintercept = c(-logfc_thresh, logfc_thresh), linetype = "dashed", color = "black") +
        theme(
          legend.position = "bottom",
          legend.title = element_blank(),
          plot.title = element_text(face = "bold", hjust = 0.5),
          plot.subtitle = element_text(hjust = 0.5)
        )
      
      # Add labels
      top_genes <- plot_data %>% filter(Significance != "Non-Significant") %>% top_n(40, wt = abs(avg_log2FC))
      if(nrow(top_genes) > 0){
        p <- p + geom_text_repel(data = top_genes, aes(label = gene), size = 3, color = "black", max.overlaps = 20)
      }
      
      # Save as JPEG 
      ggsave(plot_file, plot = p, width = 8, height = 8.15, dpi = 600, device = "jpeg", bg = "white")
    }
  }
}

In [ ]:
DimPlot(
  Seurat, 
  reduction = "umap", 
  # group.by = "CellType", 
  label = TRUE, 
  repel = TRUE,
  # label.box = TRUE,
  label.size = 4
)

ggsave( 
  "Clusters.png",
  plot   = last_plot(),
  width  = 40,
  height = 26.5,
  units  = "cm",
  dpi    = 600,
  limitsize = TRUE,
  bg     = "white"
)

In [ ]:
markers <- c(
  # Erythroid
  "Hbb-bs", "Hbb-bt", "Hbb-y", "Alas2", "Slc25a37", "Hemgn", "Gypa", "Tfrc", "Mki67", 
  # Megakaryocytes
  "Pf4", "Gp1ba", "Ppbp", "Treml1", 
  # Neutrophils
  "Hcar2", "S100a9", "Hdc", "Csf3r", "Acod1", 
  # uNK
  "Gzme", "Gzmc", "Prf1", "Fcrl6", 
  # T-cells
  "Itk", "Lck", "Cd2", "Ptpn22", "Gimap6", 
  # Monocytes
  "Ifngr1", "Dock2", "Ptprc", "Spi1", "Ly6e", 
  # Hofbauer / Macro
  "Mrc1", "C1qb", "C1qc", "Apoe", "Lyz2","Arg1",
  "H2-Aa", "H2-Ab1", "Cd74", "Cd83", "Cytip", 
  "Cybb", "Mafb", "Fcgr1", "Ifi204", "Mefv", 
  "Mmp12", "Trem2", "Spp1", "Lpl", "Gpnmb", 
  # Maternal
  "Pigr", "Muc4", "Lcn2", "Wfdc2", "Serpina1e", 
  "Prl8a2", "Htra1", "Erv3", "Procr", "Adamts5", 
  "A2m", "Des", "Htra3", "Lbp", "Masp1", 
  "Acta2", "Wnt2", "Tagln", "Itm2a", "Col4a5", 
  "Dpt", "Penk", "Pdgfra", "Igfbp6", "Sfrp4", 
  # Trophoblasts
  "Gjb3", "Ldoc1", "Tfap2c", "Rhox6", "Peg10", 
  "Prl8a8", "Ceacam12", "Psg28", "Tpbpa", "Prl7a2", 
  "Sult1e1", "Prl7b1", "Prl2a1", "Pappa2", 
  "Pnoc", "Rarres1", "Mucl3", "Lamc2", 
  "Gjb2", "Dlx3", "Tead3", "Snap91", 
  # Fetal Labyrinth
  "Lrig3", "Fndc3c1", "Klf12", "Robo2", "Kif26b", "Mdk", 
  "Grip1", "Ltbp1", "Nfia", "Zfp704", "Ebf1", "Arhgap42",
  "Plvap", "Tie1", "Emcn", "Adgrl4", "Pecam1"
)

DefaultAssay(Seurat) <- "SCT"
# DefaultAssay(Seurat) <- "Isoform"
DotPlot(
    Seurat,
    features = unique(markers),
    group.by = "CellType",
    dot.scale = 8, 
    assay = "SCT"
) + 
  scale_color_viridis() + 
  RotatedAxis()

ggsave(
  filename = "DotPlot.jpeg", 
  plot = last_plot(),
  width = 38, 
  height = 8, 
  dpi = 600  
)

In [ ]:
library(dplyr)
library(ggplot2)
library(stringr) 

meta <- Seurat@meta.data

# Loop over each BioGroup
for (bg in unique(meta$BioGroup)) {

  cell_counts <- meta %>%
    dplyr::filter(BioGroup == bg) %>%
    dplyr::count(CellType) %>%
    dplyr::mutate(
      Percent = n / sum(n) * 100
    )

  p <- ggplot(cell_counts, aes(x = str_wrap(CellType, width = 15), y = Percent, fill = CellType)) +
    geom_col() +
    geom_text(
      aes(label = sprintf("%.2f%%", Percent)),
      vjust = -0.5,
      size = 3.5
    ) +
    theme_classic() +
    theme(
      legend.position = "none",
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) + 
    ggtitle(paste("Cell Percentage -", bg)) +
    xlab("Cell Type") +
    ylab("Percentage (%)")

  ggsave(
    filename = paste0("Cell_Percentage_", bg, ".png"),
    plot     = p,
    width    = 40,
    height   = 20,
    units    = "cm",
    dpi      = 600,
    bg       = "white"
  )
}

In [ ]:
cell_counts <- as.data.frame(table(Seurat$CellType))
colnames(cell_counts) <- c("CellType", "Count")

p <- ggplot(cell_counts, aes(x = str_wrap(CellType, width = 15), y = Count, fill = CellType)) +
  geom_col() +  
  geom_text(aes(label = Count), vjust = -0.5, size = 3.5) +
  theme_classic() +
  theme(legend.position = "none") + 
  xlab("Cell Type")

# Save the plot
ggsave( 
  "Cell Count per Cluster.png",
  plot   = p,
  width  = 42,
  height = 20,
  units  = "cm",
  dpi    = 600,
  limitsize = TRUE,
  bg     = "white"
)

In [ ]:
library(dplyr)
library(ggplot2)
library(stringr) 


Idents(Seurat) <- "CellTypeDeep"
meta <- Seurat@meta.data

# Loop over each BioGroup
for (bg in unique(meta$BioGroup)) {

  cell_counts <- meta %>%
    dplyr::filter(BioGroup == bg) %>%
    dplyr::count(CellTypeDeep) %>%
    dplyr::mutate(
      Percent = n / sum(n) * 100
    )

  p <- ggplot(cell_counts, aes(x = str_wrap(CellTypeDeep, width = 15), y = Percent, fill = CellTypeDeep)) +
    geom_col() +
    geom_text(
      aes(label = sprintf("%.2f%%", Percent)),
      vjust = -0.5,
      size = 3.5
    ) +
    theme_classic() +
    theme(
      legend.position = "none",
      axis.text.x = element_text(angle = 45, hjust = 1)
    ) + 
    ggtitle(paste("Cell Percentage -", bg)) +
    xlab("Cell Type") +
    ylab("Percentage (%)")

  ggsave(
    filename = paste0("Cell_Percentage_Deep_", bg, ".png"),
    plot     = p,
    width    = 40,
    height   = 20,
    units    = "cm",
    dpi      = 600,
    bg       = "white"
  )
}

In [ ]:
DimPlot(Seurat, group.by = "Batch", label = FALSE, repel = TRUE, reduction = "umap")

ggsave( 
  "Batch.png",
  plot   = last_plot(),
  width  = 33,
  height = 27,
  units  = "cm",
  dpi    = 600,
  limitsize = TRUE,
  bg     = "white"
)

In [ ]:
DimPlot(Seurat, group.by = "BioGroup", label = FALSE, repel = TRUE, reduction = "umap")

ggsave( 
  "BioGroup.png",
  plot   = last_plot(),
  width  = 31,
  height = 27,
  units  = "cm",
  dpi    = 600,
  limitsize = TRUE,
  bg     = "white"
)

In [ ]:
library(ggplot2)
library(ggalluvial)
library(dplyr)

plot_data <- Seurat@meta.data %>%
  group_by(BioGroup, CellType) %>%
  tally() %>%
  group_by(BioGroup) %>%
  mutate(pct = n / sum(n) * 100) %>%
  ungroup()

ggplot(plot_data, 
       aes(x = BioGroup, y = pct, fill = CellType, 
           stratum = CellType, alluvium = CellType)) +
  geom_flow(alpha = 0.6, width = 0.3) + 
  geom_stratum(width = 0.3, color = "white", size = 0.2) +
  scale_y_continuous(expand = c(0, 0), labels = function(x) paste0(x, "%")) +
  theme_minimal() +
  theme(
    panel.grid.major.y = element_blank(),
    panel.grid.minor = element_blank(),
    axis.text = element_text(color = "black", size = 10),
    legend.position = "right"
  ) +
  labs(x = "Gestation Day / Pregnancy Type", y = "Composition (%)", fill = "Cell Type") +
  coord_flip()

ggsave(
  "Horizontal_Alluvial_Wide.png",
  plot   = last_plot(),
  width  = 45,  
  height = 10,  
  units  = "cm",
  dpi    = 600,
  bg     = "white"
)

In [ ]:
library(ggplot2)
library(ggalluvial)
library(dplyr)

plot_data <- Seurat@meta.data %>%
  group_by(BioGroup, CellTypeDeep) %>%
  tally() %>%
  group_by(BioGroup) %>%
  mutate(pct = n / sum(n) * 100) %>%
  ungroup()

ggplot(plot_data, 
       aes(x = BioGroup, y = pct, fill = CellTypeDeep, 
           stratum = CellTypeDeep, alluvium = CellTypeDeep)) +
  geom_flow(alpha = 0.6, width = 0.3) + 
  geom_stratum(width = 0.3, color = "white", size = 0.2) +
  scale_y_continuous(expand = c(0, 0), labels = function(x) paste0(x, "%")) +
  theme_minimal() +
  theme(
    panel.grid.major.y = element_blank(),
    panel.grid.minor = element_blank(),
    axis.text = element_text(color = "black", size = 10),
    legend.position = "right"
  ) +
  labs(x = "Gestation Day / Pregnancy Type", y = "Composition (%)", fill = "Cell Type") +
  coord_flip()

ggsave(
  "Horizontal_Alluvial_Wide_Deep.png",
  plot   = last_plot(),
  width  = 45,  
  height = 10,  
  units  = "cm",
  dpi    = 600,
  bg     = "white"
)

In [ ]:
# Cluster per group
ggplot(Seurat@meta.data, aes(x=BioGroup, fill=CellType)) + geom_bar(position = "fill")

ggsave( 
  "Cluster per group.png",
  plot   = last_plot(),
  width  = 20,
  height = 50,
  units  = "cm",
  dpi    = 600,
  limitsize = TRUE,
  bg     = "white"
)

In [ ]:
library(ggplot2)
library(stringr)

cell_counts <- as.data.frame(table(Seurat$CellType))
colnames(cell_counts) <- c("CellType", "Count")

cell_counts$Percent <- cell_counts$Count / sum(cell_counts$Count) * 100

p <- ggplot(cell_counts, aes(x = CellType, y = Percent, fill = CellType)) +
  geom_col() + 
  
  geom_text(aes(label = sprintf("%.2f%%", Percent)), 
            vjust = -0.5, size = 3.5) +
   scale_x_discrete(labels = function(x) str_wrap(x, width = 15)) +
  theme_classic() +
 theme(
    axis.text.x = element_text(color = "black", lineheight = 0.8, vjust = 1),
    legend.position = "none", # Hiding the legend since the x-axis now clearly labels everything
    plot.margin = margin(t = 10, r = 10, b = 20, l = 10) # Add a little bottom padding
  )

# Explicitly save the 'p' object rather than last_plot() to be safe
ggsave(
  "Cell Percentage per Cluster.png",
  plot   = p,
  width  = 50,
  height = 20,
  units  = "cm",
  dpi    = 600,
  limitsize = TRUE,
  bg     = "white"
)

# Isoform per BioGroup

In [ ]:
seurat_obj <- readRDS("Seurat_Deep_Clusters_Annotated.rds")

In [ ]:
seurat_obj$CellTypeDeepGroup <- paste(seurat_obj$CellTypeDeep, seurat_obj$BioGroup, sep = "_")

In [ ]:
# Idents(seurat_obj) <- "CellType"
# Idents(seurat_obj) <- "CellTypeDeep"
Idents(seurat_obj) <- "CellTypeDeepGroup"

In [ ]:
# rm(Seurat_Immune)
# rm(Seurat_Other)
# rm(Seurat_Placental)
# gc()

In [ ]:
immune_cells <- c(
  "APC-like Erythroid Cells","Macrophages (APCs)","Macrophages (Remodeling)", "Macrophages (Inflammatory)",  
  "Hofbauer Cells", "Monocytes", 
  "T-cells", 
  "NK-cell-like Erythroid Cells","uNK-cells", 
  "Neutrophil-like Erythroid Cells","Neutrophils"
  
)

ery_platelet <- c( "Erythroid Cells (Mature)", "Erythroid Cells (Fetal)","Megakaryocytes")

placental_cells <- c(
  "Labyrinthine Mesenchyme", "Syncytiotrophoblasts", "Giant Cells", 
  "Glycogen Trophoblasts", "Spongiotrophoblasts", "Labyrinth Trophoblast"
)

other_cells <- c(
  "Endothelial Cells", "Pericytes", "Fibroblasts", "Myofibroblasts", 
  "Decidua Early", "Decidua Late", "Glandular Epithelium"
)

biogroup_order <- c(
  "E19.5 Semi-Allo", 
  "E19.5 Syn", 
  "E12.5 Syn", 
  "E12.5 Semi-Allo"
)

Seurat_Immune <- subset(seurat_obj, subset = CellTypeDeep %in% immune_cells)
Seurat_Ery_Platelet <- subset(seurat_obj, subset = CellTypeDeep %in% ery_platelet)
Seurat_Placental <- subset(seurat_obj, subset = CellTypeDeep %in% placental_cells)
Seurat_Other <- subset(seurat_obj, subset = CellTypeDeep %in% other_cells)

apply_custom_order <- function(obj, ordered_cell_types, biogroup_order) {
  
  target_levels <- character()
  for (ct in ordered_cell_types) {
    target_levels <- c(target_levels, paste(ct, biogroup_order, sep = "_"))
  }
  
  existing_groups <- unique(obj$CellTypeDeepGroup)
  final_levels <- target_levels[target_levels %in% existing_groups]
  
  # Set the factor levels
  obj$CellTypeDeepGroup <- factor(obj$CellTypeDeepGroup, levels = final_levels)
  
  Idents(obj) <- "CellTypeDeepGroup"
  
  return(obj)
}

Seurat_Immune <- apply_custom_order(Seurat_Immune, immune_cells, biogroup_order)
Seurat_Ery_Platelet <- apply_custom_order(Seurat_Ery_Platelet, ery_platelet, biogroup_order)
Seurat_Placental <- apply_custom_order(Seurat_Placental, placental_cells, biogroup_order)
Seurat_Other <- apply_custom_order(Seurat_Other, other_cells, biogroup_order)

rm(seurat_obj)
gc()

In [ ]:
library(Matrix)

cytokines_isoforms_multi <- function(obj_list, min_pct = 0.25) {
  
  message("Scanning for Cytokine/Chemokine isoforms across multiple objects...")
  
  patterns <- c("^Ccl[0-9]", "^Cxcl[0-9]", "^Xcl", "^Cx3cl", 
                "^Il[0-9]", "^Tnf", "^Ifn", "^Csf", "^Tgfb")
  
  global_keep_features <- c()
  
  for (obj_name in names(obj_list)) {
    obj <- obj_list[[obj_name]]
    message(sprintf("\nProcessing %s", obj_name))
    
    DefaultAssay(obj) <- "Isoform"
    all_isoforms <- rownames(obj[["Isoform"]])
    
    target_features <- unique(unlist(lapply(patterns, function(p) grep(p, all_isoforms, value = TRUE))))
    
    if (length(target_features) == 0) {
      warning(sprintf("No cytokine isoforms found in %s!", obj_name))
      next
    }
    
    cell_types <- unique(obj$CellTypeDeep)
    cell_types <- cell_types[!is.na(cell_types)]
    
    obj_keep_features <- c()
    
    for (ct in cell_types) {
      # Identify all cells belonging to this specific Cell Type across all conditions
      cells_in_ct <- colnames(obj)[obj$CellTypeDeep == ct]
      
      if (length(cells_in_ct) == 0) next
      
      # Extract raw counts for target isoforms only for this cell type
      sub_mat <- GetAssayData(obj, assay = "Isoform", layer = "counts")[target_features, cells_in_ct, drop = FALSE]
      
      # Calculate percentage of cells with non-zero expression 
      pct_expressed <- Matrix::rowSums(sub_mat > 0) / length(cells_in_ct)
      
      # Identify features passing the threshold
      passed_threshold <- names(pct_expressed)[pct_expressed >= min_pct]
      
      # Add to the object's specific list
      obj_keep_features <- union(obj_keep_features, passed_threshold)
      
      # Early exit if we already found everything
      if (length(obj_keep_features) == length(target_features)) break
    }
    
    message(sprintf("Found %d active isoforms in %s.", length(obj_keep_features), obj_name))
    
    # Add this object's active features to the global list
    global_keep_features <- union(global_keep_features, obj_keep_features)
  }
  
  message(sprintf("\nFinal pooled list: %d unique active isoforms across all objects.", length(global_keep_features)))
  
  if (length(global_keep_features) == 0) return(NULL)
  
  # Group 1: Interleukins
  il <- list("Interleukins" = unique(grep("^Il", global_keep_features, value = TRUE)))
  il <- il[sapply(il, length) > 0] 
  
  # Group 2: Chemokines
  cc <- list(
    "Chemokines (CC)"  = unique(grep("^Ccl", global_keep_features, value = TRUE)),
    "Chemokines (CXC)" = unique(grep("^Cxcl", global_keep_features, value = TRUE))
  )
  cc <- cc[sapply(cc, length) > 0]
  
  # Group 3: Interferons, and TGF family
  if_tgf <- list(
    "Interferons"  = unique(grep("^Ifn", global_keep_features, value = TRUE)),
    "TGF-beta"     = unique(grep("^(Tgfb)", global_keep_features, value = TRUE))
  )
  if_tgf <- if_tgf[sapply(if_tgf, length) > 0]
  
  # Group 4: Other factors
  other <- list(
    "TNF Superfamily" = unique(grep("^Tnf", global_keep_features, value = TRUE)),
    "Csf"             = unique(grep("^(Csf)", global_keep_features, value = TRUE))
  )
  other <- other[sapply(other, length) > 0]
  
  return(list(il = il, cc = cc, if_tgf = if_tgf, other = other))
}


my_seurat_list <- list(
  "Immune"    = Seurat_Immune,
  "Megakaryocyte-Erythroid"    = Seurat_Ery_Platelet, 
  "Placental" = Seurat_Placental,
  "Other"     = Seurat_Other
)

cytokine_data <- cytokines_isoforms_multi(my_seurat_list, min_pct = 0.25)

In [ ]:
library(stringr)

if (!is.null(cytokine_data)) {
    
    sort_ligands_receptors <- function(features) {
      if (length(features) == 0) return(features)
      base_genes <- sub("-.*", "", features)
      
      is_receptor <- grepl("r", base_genes)
      
      # Split into ligands and receptors
      ligands <- features[!is_receptor]
      receptors <- features[is_receptor]
      
      # Apply natural alphanumeric sorting
      ligands_sorted <- str_sort(ligands, numeric = TRUE)
      receptors_sorted <- str_sort(receptors, numeric = TRUE)
      
      # Combine 
      return(c(ligands_sorted, receptors_sorted))
    }
    
    # Apply the Function
    il_genes     <- lapply(cytokine_data$il, sort_ligands_receptors)
    cc_genes     <- lapply(cytokine_data$cc, sort_ligands_receptors)
    if_tgf_genes <- lapply(cytokine_data$if_tgf, sort_ligands_receptors)
    other_genes  <- lapply(cytokine_data$other, sort_ligands_receptors)
    
    print("Lists extracted and sorted successfully (Ligands first -> Receptors).")
}

In [ ]:
group_col <- "BioGroup" 

umaps_split <- lapply(names(my_seurat_list), function(obj_name) {
  
  # Extract the current Seurat object
  seu <- my_seurat_list[[obj_name]]
  
  # Convert the group column to a factor to enforce the desired order
  seu@meta.data[[group_col]] <- factor(seu@meta.data[[group_col]], 
                                       levels = biogroup_order)
  
  # Generate the UMAP plot
  p <- DimPlot(seu, 
               reduction = "umap", 
               group.by = "CellTypeDeep", 
               split.by = group_col) +
    ggtitle(obj_name) +
    theme(
      plot.title = element_text(hjust = 0.5, face = "bold", size = 16),
      aspect.ratio = 1 # Forces each individual UMAP panel to be perfectly square
    )
  
  return(p)
})

# Assign names to the list elements for easy access
names(umaps_split) <- names(my_seurat_list)

# Loop through the generated plots and save them
for (obj_name in names(umaps_split)) {
  
  # Define the filename (e.g., "UMAP_Immune.png")
  file_name <- paste0("UMAP_", obj_name, ".png")
  
  # Save the plot with 600 dpi
  ggsave(filename = file_name, 
         plot = umaps_split[[obj_name]], 
         width = 20, 
         height = 5, 
         dpi = 600, 
         bg = "white")
}

In [ ]:
gene_lists_collection <- list(
  "il_genes"     = il_genes,
  "if_tgf_genes" = if_tgf_genes,
  "cc_genes"     = cc_genes,
  "other_genes"  = other_genes
)

In [ ]:
library(Seurat)
library(ggplot2)
library(viridis)

dot_spacing <- 0.225

for (obj_name in names(my_seurat_list)) {
  
  seu <- my_seurat_list[[obj_name]]
  
  for (gene_list_name in names(gene_lists_collection)) {
    
    current_genes <- gene_lists_collection[[gene_list_name]]
    
    p_base <- DotPlot(
      seu,
      features = current_genes,
      assay = "Isoform"
    )
    
    p_custom <- ggplot(p_base$data, aes(x = features.plot, y = id)) +
      geom_point(aes(size = pct.exp, color = avg.exp.scaled)) +
      scale_color_gradient2(
        low = "blue", 
        mid = "white", 
        high = "red", 
        midpoint = 0, 
        name = "Average\nExpression"
      ) +
      scale_size_continuous(range = c(0, 8), name = "Percent\nExpressed") +
      theme_classic() + 
      theme(
        axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1, color = "black"),
        axis.text.y = element_text(color = "black"),
        axis.title = element_blank(),
        
        plot.margin = margin(t = 10, r = 10, b = 10, l = 60, unit = "pt")
      ) +
      coord_fixed(ratio = 1, clip = "off") 
    
    n_genes <- length(unique(p_base$data$features.plot))
    n_celltypes <- length(unique(p_base$data$id))
    
    calc_width <- (n_genes * dot_spacing) + 5.5
    calc_height <- (n_celltypes * dot_spacing) + 3.5
    
    file_name <- paste0("DotPlot_", obj_name, "_", gene_list_name, ".jpeg")
    
    suppressWarnings(
      ggsave(
        filename = file_name, 
        plot = p_custom,
        width = calc_width, 
        height = calc_height, 
        dpi = 600,
        bg = "white",
        limitsize = FALSE
      )
    )
  }
}

In [ ]:
library(Seurat)
library(ggplot2)
library(ggrepel) 

all_genes <- unique(unlist(gene_lists_collection, use.names = FALSE))

# Define the 4 valid comparisons
comparisons <- list(
  c("E19.5 Semi-Allo", "E19.5 Syn"),       
  c("E12.5 Semi-Allo", "E12.5 Syn"),       
  c("E19.5 Semi-Allo", "E12.5 Semi-Allo"), 
  c("E19.5 Syn",       "E12.5 Syn")        
)

# Thresholds
p_val_cutoff <- 0.0005
log2fc_cutoff <- 1 

dgelist <- list()

for (obj_name in names(my_seurat_list)) {
  
  seu <- my_seurat_list[[obj_name]]
  Idents(seu) <- "BioGroup"
  
  for (comp in comparisons) {
    ident1 <- comp[1]
    ident2 <- comp[2]
    
    if (!(ident1 %in% levels(Idents(seu))) || !(ident2 %in% levels(Idents(seu)))) {
      next
    }
    
    # Calculate Differential Expression
    deg_results <- tryCatch({
      FindMarkers(
        seu, 
        ident.1 = ident1, 
        ident.2 = ident2, 
        features = all_genes, 
        assay = "Isoform",
        logfc.threshold = 0, 
        min.pct = 0          
      )
    }, error = function(e) {
      return(NULL)
    })
    
    if (is.null(deg_results) || nrow(deg_results) == 0) next
    
    # Clean up the data
    deg_results$gene <- rownames(deg_results)
    
    min_p_val <- min(deg_results$p_val_adj[deg_results$p_val_adj > 0])
    deg_results$p_val_adj[deg_results$p_val_adj == 0] <- min_p_val
    
    deg_results <- subset(deg_results, p_val_adj < 1)
    
    if (nrow(deg_results) == 0) next
    
    # Label significance
    deg_results$Significance <- "Not Significant"
    up_ident1_label <- paste("Up in", ident1)
    up_ident2_label <- paste("Up in", ident2)
    
    deg_results$Significance[deg_results$avg_log2FC > log2fc_cutoff & deg_results$p_val_adj < p_val_cutoff] <- up_ident1_label
    deg_results$Significance[deg_results$avg_log2FC < -log2fc_cutoff & deg_results$p_val_adj < p_val_cutoff] <- up_ident2_label
    
    # Calculate symmetric X-axis limits
    max_x <- max(abs(deg_results$avg_log2FC), na.rm = TRUE)
    x_limit <- max_x * 1.05 
    
    # Build the Volcano Plot
    p <- ggplot(deg_results, aes(x = avg_log2FC, y = -log10(p_val_adj), color = Significance)) +
      geom_point(alpha = 0.8, size = 2.5) +
      scale_color_manual(
        values = setNames(
          c("firebrick3", "navy", "grey80"),
          c(up_ident1_label, up_ident2_label, "Not Significant")
        )
      ) +
      scale_x_continuous(
        limits = c(-x_limit, x_limit),
        breaks = seq(-ceiling(x_limit), ceiling(x_limit), by = 1)
      ) + 
      geom_vline(xintercept = c(-log2fc_cutoff, log2fc_cutoff), linetype = "dashed", color = "black", alpha = 0.5) +
      geom_hline(yintercept = -log10(p_val_cutoff), linetype = "dashed", color = "black", alpha = 0.5) +
      
      geom_text_repel(
        data = subset(deg_results, Significance != "Not Significant"),
        aes(label = gene),
        size = 3.5,
        max.overlaps = 30, 
        box.padding = 0.5,
        segment.linewidth = 0.2, 
        segment.alpha = 0.4,     
        show.legend = FALSE 
      ) +
      theme_classic() +
      labs(
        title = paste(obj_name, ":", ident1, "vs", ident2),
        subtitle = "Targeted Cytokine DGE",
        x = expression("Average " * log[2] * "(Fold Change)"),
        y = expression("-log"[10] * "(Adjusted P-value)")
      ) +
      theme(
        panel.border = element_rect(colour = "black", fill = NA, linewidth = 1),
        axis.line = element_blank(), 
        
        plot.title = element_text(hjust = 0.5, face = "bold", size = 14),
        plot.subtitle = element_text(hjust = 0.5, size = 12),
        legend.position = "bottom",
        legend.title = element_blank()
      )
    
    # Save the plot
    safe_ident1 <- gsub(" ", "_", ident1)
    safe_ident2 <- gsub(" ", "_", ident2)
    file_name <- paste0("Volcano_", obj_name, "_", safe_ident1, "_vs_", safe_ident2, ".jpeg")
    
    suppressWarnings(ggsave(filename = file_name, plot = p, width = 8, height = 8, dpi = 600, bg = "white"))
    
    # DGE
    sig_results <- subset(deg_results, Significance != "Not Significant")
    
    if (nrow(sig_results) > 0) {
      sig_results$Compartment <- obj_name
      sig_results$Comparison <- paste(ident1, "vs", ident2)
      
      list_name <- paste(obj_name, safe_ident1, safe_ident2, sep = "_")
      dgelist[[list_name]] <- sig_results
    }
  }
}

if (length(dgelist) > 0) {
  # Combine
  dgedf <- do.call(rbind, dgelist)
  
  cols_order <- c("Compartment", "Comparison", "gene", "avg_log2FC", "p_val_adj", "pct.1", "pct.2", "Significance")
  dgedf <- dgedf[, intersect(cols_order, colnames(dgedf))]
  
  # Sort by Compartment, then Comparison, then Fold Change
  dgedf <- dgedf[order(dgedf$Compartment, dgedf$Comparison, -abs(dgedf$avg_log2FC)), ]
  
  rownames(dgedf) <- NULL
  
  write.csv(dgedf, file = "Targeted_Cytokine_DGE.csv", row.names = FALSE)
  print(paste("Done. CSV generated with", nrow(dgedf), "total significant isoforms."))
} else {
  print("No significant hits were found across any comparisons.")
}

In [ ]:
# install.packages(c("ggVennDiagram", "stringr"))

In [ ]:
library(ggVennDiagram)
library(ggplot2)
library(dplyr)
library(stringr)

df <- read.csv("Targeted_Cytokine_DGE.csv")

plot_named_venn <- function(data, direction, file_name, fill_palette, edge_color) {
  
  if (direction == "UP") {
    sub_df <- data %>% filter(avg_log2FC > 0)
  } else {
    sub_df <- data %>% filter(avg_log2FC < 0)
  }
  
  venn_lists <- split(sub_df$gene, sub_df$Comparison)
  venn_obj <- process_data(Venn(venn_lists))
  
  regions <- venn_region(venn_obj)
  region_labels <- venn_regionlabel(venn_obj)
  
  region_labels$formatted_labels <- sapply(regions$item, function(genes) {
    if(length(genes) == 0) return("")
    wrap_w <- ifelse(length(genes) > 15, 26, 18)
    paste(strwrap(paste(genes, collapse = ", "), width = wrap_w), collapse = "\n")
  })
  
  region_labels$label_size <- sapply(regions$item, function(genes) {
    n <- length(genes)
    if (n == 0) return(0)
    if (n > 25) return(1.2)
    if (n > 18) return(2.5)   
    if (n > 7) return(2.6)   
    if (n > 5)  return(4)
    if (n == 1)  return(4)
    return(4 )               
  })
  
  set_labels <- venn_setlabel(venn_obj)
  set_labels$name <- str_wrap(set_labels$name, width = 16)
  
  p <- ggplot() +
    geom_polygon(aes(X, Y, fill = count, group = id), 
                 data = venn_regionedge(venn_obj),
                 color = "white", linewidth = 1) +
    
    geom_path(aes(X, Y, group = id), 
              data = venn_setedge(venn_obj), 
              color = edge_color, linewidth = 0.8) +
    
    # Draw group names
    geom_text(aes(X, Y, label = name), 
              data = set_labels, 
              size = 4.5, fontface = "bold", color = edge_color, lineheight = 0.9) +
    
    # Draw gene names 
    geom_text(aes(X, Y, label = formatted_labels, size = label_size), 
              data = region_labels, 
              lineheight = 0.85, color = "black", fontface = "bold") +
    
    scale_size_identity() +
    scale_fill_distiller(palette = fill_palette, direction = 1, name = "Gene Count") +
    
    coord_equal() +
    
    theme_void() +
    theme(
      plot.margin = margin(t = 1, r = 1, b = 1, l = 1, unit = "cm"),
      plot.background = element_rect(fill = "white", color = NA),
      legend.position = "bottom" 
    )
  
  suppressWarnings(
    ggsave(file_name, plot = p, width = 11.5, height = 11.5, dpi = 1800, bg = "white")
  )
  
  message("Successfully saved: ", file_name)
  return(p)
}

plot_named_venn(
  data = df, 
  direction = "UP", 
  file_name = "Venn_Diagram_Named_UP.jpeg", 
  fill_palette = "Reds", 
  edge_color = "firebrick"
)

plot_named_venn(
  data = df, 
  direction = "DOWN", 
  file_name = "Venn_Diagram_Named_DOWN.jpeg", 
  fill_palette = "Blues", 
  edge_color = "navy"
)

# DGE ORA

In [ ]:
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

# BiocManager::install("clusterProfiler")
BiocManager::install("org.Mm.eg.db")

In [ ]:
# Load necessary libraries
library(dplyr)
library(purrr)
library(stringr)
library(readr)
library(clusterProfiler)
library(org.Mm.eg.db) 
library(ggplot2)

select <- dplyr::select

dge_dir <- "./DGE"
all_csv_files <- list.files(dge_dir, pattern = "\\.csv$", recursive = TRUE, full.names = TRUE)

master_df <- map_df(all_csv_files, function(file_path) {
  
  parts <- unlist(str_split(file_path, "/"))
  assay <- parts[length(parts) - 1]
  file_name <- str_remove(parts[length(parts)], "\\.csv$")
  name_parts <- str_split_fixed(file_name, "_", 2)
  cell_type <- name_parts[1]
  comparison <- name_parts[2]
  
  df <- read_csv(file_path, 
                 show_col_types = FALSE,
                 col_types = cols(
                   p_val = col_double(),
                   avg_log2FC = col_double(),
                   pct.1 = col_double(),
                   pct.2 = col_double(),
                   p_val_adj = col_double(),
                   p_val_adj_plot = col_double(),
                   gene = col_character(),
                   Significance = col_character()
                 ))
  
  if(nrow(df) > 0) {
    df$Assay <- assay
    df$CellTypeDeep <- cell_type
    df$Comparison <- comparison
  }
  return(df)
})

# Save Full DF
write.csv(master_df, "DGE_All_Assays.csv", row.names = FALSE)

# ANALYSIS 
sct_hits <- master_df %>% 
  filter(Assay == "SCT" & Significance %in% c("Up", "Down"))

# Create unique analysis list: CellType + Comparison + Direction
analyses <- sct_hits %>%
  select(CellTypeDeep, Comparison, Significance) %>%
  distinct()

# PER-CELL-TYPE GO ORA
dir.create("DGE", showWarnings = FALSE)

for (i in 1:nrow(analyses)) {
  ct   <- analyses$CellTypeDeep[i]
  comp <- analyses$Comparison[i]
  sig  <- analyses$Significance[i]
  
  # Get gene list for this specific cluster
  genes_to_test <- sct_hits %>%
    filter(CellTypeDeep == ct, Comparison == comp, Significance == sig) %>%
    pull(gene)
  
  # Skip if list is too small
  if (length(genes_to_test) < 5) {
    message(sprintf("Skipping %s | %s | %s - too few genes (%d)", ct, comp, sig, length(genes_to_test)))
    next
  }
  
  message(sprintf("Processing GO: %s | %s | %s", ct, comp, sig))
  
  # Run Enrichment
  go_res <- tryCatch({
    enrichGO(gene          = genes_to_test,
             OrgDb         = org.Mm.eg.db,
             keyType       = "SYMBOL",
             ont           = "BP",
             pAdjustMethod = "BH",
             pvalueCutoff  = 0.05,
             qvalueCutoff  = 0.2,
             readable      = TRUE)
  }, error = function(e) return(NULL))
  
  # Plot and Save
  if (!is.null(go_res) && nrow(go_res) > 0) {
    
    # Create subfolders per cell type to keep it organized
    ct_folder <- file.path("DGE", gsub("/", "_", ct))
    dir.create(ct_folder, showWarnings = FALSE)
    
    p <- dotplot(go_res, showCategory = 10) +
      ggtitle(paste(ct, "-", sig), subtitle = comp) +
      theme_minimal() +
      theme(plot.title = element_text(size = 12, face = "bold"),
            axis.text.y = element_text(size = 8))
    
    safe_name <- paste0(gsub(" ", "", comp), "_", sig, ".jpg")
    ggsave(file.path(ct_folder, safe_name), plot = p, width = 8, height = 6, dpi = 300)
    
  } else {
    message("  -> No enrichment found.")
  }
}

message("GO ORA complete!")

# Erythroid Cell Subclustering

In [ ]:
# Subset Erythroid Cells

message("Subsetting Erythroid Cells...")

Ery_Seurat <- subset(Seurat, idents = c("Erythroid Cells"))

Ery_Seurat

Ery_Seurat@meta.data

In [ ]:
saveRDS(Ery_Seurat, file = "Erythroid_Cells.rds")

# Save as h5ad

In [ ]:
library(Seurat)
library(BPCells)

message("Loading and updating object...")
# UpdateSeuratObject ensures v4 objects work with v5 functions (like Layers/LayerData)
seurat_obj <- readRDS("Seurat_Deep_Clusters_Annotated.rds")
seurat_obj <- UpdateSeuratObject(seurat_obj)

assays_to_keep <- c("SCT", "Isoform")

for (assay in assays_to_keep) {
  # names() is safer than Assays() in buggy environments
  if (assay %in% names(seurat_obj)) {
    
    # Check if 'data' layer exists (v5 style)
    available_layers <- Layers(seurat_obj[[assay]])
    
    if ("data" %in% available_layers) {
      message(paste("Processing", assay, "data layer..."))
      
      mat <- LayerData(seurat_obj, assay = assay, layer = "data")
      
      # BPCells check: conversion to IterableMatrix if it's still a standard sparse matrix
      if (!inherits(mat, "IterableMatrix")) {
        message(paste("  -> Converting", assay, "to IterableMatrix..."))
        mat <- as(mat, "IterableMatrix")
      }
      
      message(paste("  -> Streaming", assay, "to HDF5..."))
      write_matrix_10x_hdf5(mat, path = paste0(assay, "_data.h5"))
      
    } else {
      message(paste("Skipping", assay, "- 'data' layer not found."))
    }
  } else {
    message(paste("Skipping", assay, "- Assay not found in object."))
  }
}

message("Exporting metadata...")
write.csv(seurat_obj@meta.data, "metadata.csv", row.names = TRUE)

message("Exporting embeddings...")
for (red in names(seurat_obj@reductions)) {
  message(paste("  -> Exporting", red))
  write.csv(Embeddings(seurat_obj, red), paste0(red, "_embeddings.csv"), row.names = TRUE)
}

message("Export complete! Time to assemble in Python.")